# Import libraries

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import urllib.request
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
class CybersecModelLoader:
    """Load and test cybersecurity models"""

    def __init__(self, model_dir):
        """Initialize with the directory containing saved models"""
        self.model_dir = model_dir
        self.lr_model = None
        self.rf_model = None
        self.scaler = None
        self.label_encoders = None
        self.feature_info = None
        self.performance_info = None

    def load_models(self):
        """Load all saved models and preprocessing objects"""
        try:
            # Load models
            self.lr_model = joblib.load(f'{self.model_dir}/logistic_regression.pkl')
            self.rf_model = joblib.load(f'{self.model_dir}/random_forest.pkl')

            # Load preprocessing objects
            self.scaler = joblib.load(f'{self.model_dir}/feature_scaler.pkl')
            self.label_encoders = joblib.load(f'{self.model_dir}/label_encoders.pkl')

            # Load metadata
            with open(f'{self.model_dir}/model_info.json', 'r') as f:
                model_info = json.load(f)
                self.feature_info = model_info['feature_info']
                self.performance_info = model_info['performance']

            print(f"✅ Successfully loaded models from {self.model_dir}")
            print(f"📊 Expected features: {self.feature_info['n_features']}")
            print(f"📊 Categorical columns: {self.feature_info['categorical_columns']}")
            print(f"📊 Numerical columns: {len(self.feature_info['numerical_columns'])}")
            print(f"📊 Label encoders loaded: {list(self.label_encoders.keys())}")

            return True

        except Exception as e:
            print(f"❌ Error loading models: {str(e)}")
            return False

    def download_dataset(self):
        """Download NSL-KDD dataset files"""
        base_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
        files = ['KDDTrain+.txt', 'KDDTest+.txt']

        print("📥 Downloading NSL-KDD dataset...")

        for file in files:
            if not os.path.exists(file):
                try:
                    urllib.request.urlretrieve(base_url + file, file)
                    print(f"✅ Downloaded {file}")
                except Exception as e:
                    print(f"❌ Error downloading {file}: {str(e)}")
                    return False
            else:
                print(f"✅ {file} already exists")

        return True

    def load_and_preprocess_data(self, filename):
        """Load and preprocess NSL-KDD data"""
        # NSL-KDD column names (matching your training exactly)
        columns = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
                   'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
                   'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
                   'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
                   'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
                   'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
                   'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
                   'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
                   'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
                   'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack_type', 'difficulty']

        try:
            # Load data
            data = pd.read_csv(filename, names=columns)

            print(f"✅ Loaded {filename}: {data.shape[0]} samples, {data.shape[1]} features")
            print(f"📊 Attack types: {data['attack_type'].nunique()} unique types")
            print(f"📊 Normal vs Attack: {(data['attack_type'] == 'normal').sum()} normal, {(data['attack_type'] != 'normal').sum()} attacks")

            return data

        except Exception as e:
            print(f"❌ Error loading {filename}: {str(e)}")
            return None

    def preprocess_features(self, data):
        """Apply the same preprocessing as training"""
        try:
            # Create binary target (same as training)
            data['binary_class'] = data['attack_type'].apply(lambda x: 0 if x == 'normal' else 1)

            # Get categorical and numerical columns from saved info
            categorical_cols = self.feature_info['categorical_columns']
            numerical_cols = self.feature_info['numerical_columns']

            # Apply label encoding to categorical features
            for col in categorical_cols:
                if col in data.columns:
                    if col in self.label_encoders:
                        # Handle unseen categories
                        le = self.label_encoders[col]
                        data[col] = data[col].astype(str)

                        # Map unseen categories to a default value
                        mask = ~data[col].isin(le.classes_)
                        if mask.any():
                            print(f"⚠️  Found {mask.sum()} unseen categories in {col}")
                            data.loc[mask, col] = le.classes_[0]  # Use first class as default

                        data[col] = le.transform(data[col])
                    else:
                        print(f"⚠️  No encoder found for {col}")

            # Prepare features (numerical + categorical columns, same order as training)
            X = data[numerical_cols + categorical_cols].copy()
            y = data['binary_class']

            # Scale ONLY numerical features (same as training)
            X_scaled = X.copy()
            X_scaled[numerical_cols] = self.scaler.transform(X[numerical_cols])

            print(f"✅ Preprocessed features: {X_scaled.shape}")
            print(f"   - Numerical columns scaled: {len(numerical_cols)}")
            print(f"   - Categorical columns encoded: {len(categorical_cols)}")

            return X_scaled, y

        except Exception as e:
            print(f"❌ Error preprocessing features: {str(e)}")
            return None, None

    def evaluate_models(self, X_test, y_test):
        """Evaluate both models on test data"""
        results = {}

        print("\n🧪 Testing Models...")
        print("="*50)

        # Test Logistic Regression
        try:
            lr_pred = self.lr_model.predict(X_test)
            lr_proba = self.lr_model.predict_proba(X_test)[:, 1]

            results['logistic_regression'] = {
                'predictions': lr_pred,
                'probabilities': lr_proba,
                'accuracy': accuracy_score(y_test, lr_pred),
                'precision': precision_score(y_test, lr_pred),
                'recall': recall_score(y_test, lr_pred),
                'f1_score': f1_score(y_test, lr_pred)
            }

            print("📊 Logistic Regression Results:")
            print(f"   Accuracy:  {results['logistic_regression']['accuracy']:.4f}")
            print(f"   Precision: {results['logistic_regression']['precision']:.4f}")
            print(f"   Recall:    {results['logistic_regression']['recall']:.4f}")
            print(f"   F1-Score:  {results['logistic_regression']['f1_score']:.4f}")

        except Exception as e:
            print(f"❌ Error testing Logistic Regression: {str(e)}")

        # Test Random Forest
        try:
            rf_pred = self.rf_model.predict(X_test)
            rf_proba = self.rf_model.predict_proba(X_test)[:, 1]

            results['random_forest'] = {
                'predictions': rf_pred,
                'probabilities': rf_proba,
                'accuracy': accuracy_score(y_test, rf_pred),
                'precision': precision_score(y_test, rf_pred),
                'recall': recall_score(y_test, rf_pred),
                'f1_score': f1_score(y_test, rf_pred)
            }

            print("\n📊 Random Forest Results:")
            print(f"   Accuracy:  {results['random_forest']['accuracy']:.4f}")
            print(f"   Precision: {results['random_forest']['precision']:.4f}")
            print(f"   Recall:    {results['random_forest']['recall']:.4f}")
            print(f"   F1-Score:  {results['random_forest']['f1_score']:.4f}")

        except Exception as e:
            print(f"❌ Error testing Random Forest: {str(e)}")

        return results

    def compare_with_training(self, test_results):
        """Compare test results with original training performance"""
        print("\n📈 Performance Comparison (Training vs Test)")
        print("="*60)

        for model_name in ['logistic_regression', 'random_forest']:
            if model_name in test_results and model_name in self.performance_info:
                train_perf = self.performance_info[model_name]
                test_perf = test_results[model_name]

                print(f"\n{model_name.replace('_', ' ').title()}:")
                print(f"                Training    Test      Difference")
                print(f"   Accuracy:    {train_perf['accuracy']:.4f}    {test_perf['accuracy']:.4f}    {test_perf['accuracy'] - train_perf['accuracy']:+.4f}")
                print(f"   Precision:   {train_perf['precision']:.4f}    {test_perf['precision']:.4f}    {test_perf['precision'] - train_perf['precision']:+.4f}")
                print(f"   Recall:      {train_perf['recall']:.4f}    {test_perf['recall']:.4f}    {test_perf['recall'] - train_perf['recall']:+.4f}")
                print(f"   F1-Score:    {train_perf['f1_score']:.4f}    {test_perf['f1_score']:.4f}    {test_perf['f1_score'] - train_perf['f1_score']:+.4f}")

    def generate_detailed_report(self, X_test, y_test, results):
        """Generate detailed classification report"""
        print("\n📋 Detailed Classification Reports")
        print("="*50)

        for model_name, model_results in results.items():
            print(f"\n{model_name.replace('_', ' ').title()}:")
            print(classification_report(y_test, model_results['predictions'],
                                      target_names=['Normal', 'Attack']))

    def run_complete_test(self):
        """Run the complete testing pipeline"""
        print("🚀 Starting Cybersecurity Model Testing Pipeline")
        print("="*60)

        # Step 1: Load models
        if not self.load_models():
            return False

        # Step 2: Download dataset
        if not self.download_dataset():
            return False

        # Step 3: Load and preprocess test data
        test_data = self.load_and_preprocess_data('KDDTest+.txt')
        if test_data is None:
            return False

        # Step 4: Preprocess features
        X_test, y_test = self.preprocess_features(test_data)
        if X_test is None:
            return False

        # Step 5: Evaluate models
        results = self.evaluate_models(X_test, y_test)

        # Step 6: Compare with training performance
        self.compare_with_training(results)

        # Step 7: Generate detailed reports
        self.generate_detailed_report(X_test, y_test, results)

        print("\n✅ Testing pipeline completed successfully!")
        return True

# Usage example:
def test_saved_models(model_directory):
    """Test the saved cybersecurity models"""
    tester = CybersecModelLoader(model_directory)
    success = tester.run_complete_test()

    if success:
        print(f"\n🎯 Models from {model_directory} tested successfully!")
    else:
        print(f"\n❌ Testing failed for {model_directory}")

    return tester


In [5]:
# Example usage:
# Replace 'cybersec_models_YYYYMMDD_HHMMSS' with your actual saved model directory
if __name__ == "__main__":
    # Example model directory - replace with your actual directory name
    model_dir = "models"  # Replace with your actual directory

    # Test the models
    tester = test_saved_models(model_dir)

🚀 Starting Cybersecurity Model Testing Pipeline
✅ Successfully loaded models from models
📊 Expected features: 41
📊 Categorical columns: ['protocol_type', 'service', 'flag']
📊 Numerical columns: 38
📊 Label encoders loaded: ['protocol_type', 'service', 'flag']
📥 Downloading NSL-KDD dataset...
✅ KDDTrain+.txt already exists
✅ KDDTest+.txt already exists
✅ Loaded KDDTest+.txt: 22544 samples, 43 features
📊 Attack types: 38 unique types
📊 Normal vs Attack: 9711 normal, 12833 attacks
✅ Preprocessed features: (22544, 41)
   - Numerical columns scaled: 38
   - Categorical columns encoded: 3

🧪 Testing Models...
📊 Logistic Regression Results:
   Accuracy:  0.7539
   Precision: 0.9253
   Recall:    0.6175
   F1-Score:  0.7407

📊 Random Forest Results:
   Accuracy:  0.7666
   Precision: 0.9664
   Recall:    0.6113
   F1-Score:  0.7489

📈 Performance Comparison (Training vs Test)

Logistic Regression:
                Training    Test      Difference
   Accuracy:    0.7539    0.7539    +0.0000
   Pr